# Sprint 3 — LLM Parameter Generation

Generates building energy parameters via LLM API calls under 4 prompt
formats (F1-F4: identity / +unit / +range / +unit+range), using the
neutral `param_1...param_12` schema and LOBO ranges from Sprint 2.

**Two independent parts:**
- **Part 1** — builds and validates all prompts (leakage check,
  cross-format consistency), no API key needed. Saves to
  `analiz/sprint3_prompts/`.
- **Part 2** — calls 2 models (Llama 3.3 70B, gpt-oss-120b) × 5 seeds
  per format, with API key rotation and resumable, incremental logging.

**Climate:** Buffalo, NY (5A) for the main grid; F4 also runs on
Miami (1A) and International Falls (7) for climate robustness.

In [6]:
# ============================================================
# PART 1-A — SETUP: protocol, parameter schema, prompt builders
# (no API key needed, no network calls)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, re, json, hashlib
import pandas as pd

BASE        = '/content/drive/MyDrive/BEM-LLM'
DATA        = f'{BASE}/data'
ANALIZ      = f'{BASE}/analiz'
SPRINT2_DIR = f'{ANALIZ}/sprint2_lobo'
PROMPTS_DIR = f'{ANALIZ}/sprint3_prompts'
os.makedirs(PROMPTS_DIR, exist_ok=True)

with open(f'{DATA}/environment_sprint1.json') as f:
    MASTER_SEED = json.load(f)['master_seed']
with open(f'{ANALIZ}/protocol.json') as f:
    protocol = json.load(f)

TESTED_BUILDINGS = protocol['tested_buildings']
CLIMATES = protocol['climates']
SEEDS = protocol['seeds']

CLIMATE_LABELS = {
    '5A': 'Buffalo, NY (ASHRAE Climate Zone 5A)',
    '1A': 'Miami, FL (ASHRAE Climate Zone 1A)',
    '7':  'International Falls, MN (ASHRAE Climate Zone 7)',
}
CLIMATE_ROLE = {CLIMATES['main']: 'main_experiment'}
for c in CLIMATES['validation']:
    CLIMATE_ROLE[c] = 'weather_file_stress_test'

with open(f'{SPRINT2_DIR}/lobo_range_table.json') as f:
    lobo = json.load(f)
RANGE_TABLE = lobo['range_table']
TARGET_PARAMS = lobo['target_parameters']       # 9, not 12
CONTROL_PARAMS = lobo['control_parameters']     # 3, never shown to the model

with open(f'{SPRINT2_DIR}/control_parameters.json') as f:
    control_data = json.load(f)
CONTROL_VALUES = control_data['values_by_building']

with open(f'{SPRINT2_DIR}/lobo_range_table.json', 'rb') as f:
    lobo_hash = hashlib.sha256(f.read()).hexdigest()[:16]
print(f'lobo_range_table.json SHA-256: {lobo_hash}')

# --- Input validation: LOBO range table integrity, before anything uses it ---
assert set(RANGE_TABLE[TESTED_BUILDINGS[0]].keys()) == set(TARGET_PARAMS), \
    'lobo_range_table.json: range_table keys do not match the 9 target_parameters'

assert all(RANGE_TABLE[b][p][0] <= RANGE_TABLE[b][p][1]
           for b in TESTED_BUILDINGS for p in RANGE_TABLE[b]), \
    'lobo_range_table.json: found lo > hi in at least one building/parameter'

assert all(0 <= RANGE_TABLE[b]['window_shgc'][0] and RANGE_TABLE[b]['window_shgc'][1] <= 1
           for b in TESTED_BUILDINGS), \
    'lobo_range_table.json: window_shgc range outside [0,1]'

assert all(RANGE_TABLE[b][p][0] >= 0
           for b in TESTED_BUILDINGS for p in RANGE_TABLE[b]
           if p not in ('heating_setpoint_C', 'cooling_setpoint_C')), \
    'lobo_range_table.json: negative lower bound on a non-temperature parameter'

ref_df_check = pd.read_csv(f'{SPRINT2_DIR}/reference_parameters.csv', index_col=0)
for b in TESTED_BUILDINGS:
    for p_name in RANGE_TABLE[b]:
        lo, hi = RANGE_TABLE[b][p_name]
        own_val = ref_df_check.loc[b, p_name]
        assert not (lo == own_val or hi == own_val), \
            f'{b}/{p_name}: own reference value sits exactly on range boundary — check drop(index=bldg)'

print('LOBO range table integrity checks passed (bounds, SHGC, non-negativity, own-value exclusion, 9-param schema).')

# --- Neutral parameter schema: 9 target parameters only. Control parameters ---
# --- (infiltration, heating_efficiency, roof_u_value) are never presented to ---
# --- the model at all -- not as a question, not as a "fixed" note. ---
PARAM_SCHEMA = [
    {'id': 'param_1', 'name': 'window_u_value',
     'identity': 'How easily heat passes through the window system from indoors to outdoors (or vice versa).',
     'unit': 'This value is in W/m\u00b2K.'},
    {'id': 'param_2', 'name': 'window_shgc',
     'identity': 'The fraction of solar radiation striking the windows that enters the building as heat gain.',
     'unit': 'This value is a unitless fraction between 0 and 1.'},
    {'id': 'param_3', 'name': 'lighting_W_m2',
     'identity': 'The electrical power drawn by interior lighting, relative to the floor area it serves.',
     'unit': 'This value is in W/m\u00b2.'},
    {'id': 'param_4', 'name': 'equipment_W_m2',
     'identity': 'The electrical power drawn by plug-in and process equipment, relative to the floor area it serves.',
     'unit': 'This value is in W/m\u00b2.'},
    {'id': 'param_5', 'name': 'occupancy_m2_person',
     'identity': 'The average floor area allocated per building occupant.',
     'unit': 'This value is in m\u00b2/person.'},
    {'id': 'param_6', 'name': 'outdoor_air',
     'identity': 'The rate of outdoor air intentionally supplied for ventilation, relative to floor area.',
     'unit': 'This value is in m\u00b3/s per m\u00b2.'},
    {'id': 'param_7', 'name': 'cooling_cop',
     'identity': 'The ratio of cooling output produced by the cooling system to the electrical input it consumes.',
     'unit': 'This value is a unitless ratio (output/input).'},
    {'id': 'param_8', 'name': 'heating_setpoint_C',
     'identity': 'The indoor temperature the heating system attempts to maintain during occupied hours in cold conditions.',
     'unit': 'This value is in degrees Celsius.'},
    {'id': 'param_9', 'name': 'cooling_setpoint_C',
     'identity': 'The indoor temperature the cooling system attempts to maintain during occupied hours in hot conditions.',
     'unit': 'This value is in degrees Celsius.'},
]
assert [p['name'] for p in PARAM_SCHEMA] == TARGET_PARAMS, \
    'PARAM_SCHEMA order/content does not match lobo_range_table.json target_parameters -- keep these in sync'

NAME_TO_ID = {p['name']: p['id'] for p in PARAM_SCHEMA}
ID_TO_NAME = {p['id']: p['name'] for p in PARAM_SCHEMA}
JSON_KEY_ORDER = [p['id'] for p in PARAM_SCHEMA]     # 9 keys, not 12
PARAM_NAMES = [p['name'] for p in PARAM_SCHEMA]

def _schema_line(p):
    return f'"{p["id"]}": 0.0'

SYSTEM_PROMPT_BASE = (
    "You are a building energy expert. For each of the following 9 parameters, "
    "estimate a plausible numeric value for the specified building. "
    "Return ONLY a JSON object with exactly these keys, in this order, no explanation, "
    "no markdown, no text outside the JSON:\n"
    "{" + ", ".join(_schema_line(p) for p in PARAM_SCHEMA) + "}\n\n"
    "Parameter definitions:\n"
    + "\n".join(f'- {p["id"]}: {p["identity"]}' for p in PARAM_SCHEMA)
)

BASE_USER_PROMPT = ("Building type: {building}. Location: {location}. "
                     "Provide ASHRAE 90.1-2019 compliant energy modeling parameters.")
UNITS_CLAUSE_TEXT = " Return values in exactly the specified units."

# --- Range phrasing corrected to describe what the range actually is (cross-
# prototype, target building excluded), with per-parameter decimal precision. ---
RANGE_DECIMALS = {
    'window_u_value': 2, 'window_shgc': 2, 'lighting_W_m2': 1,
    'equipment_W_m2': 1, 'occupancy_m2_person': 1, 'outdoor_air': 4,
    'cooling_cop': 1, 'heating_setpoint_C': 1, 'cooling_setpoint_C': 1,
}

def _range_phrase(building, p):
    lo, hi = RANGE_TABLE[building][p['name']]
    d = RANGE_DECIMALS[p['name']]
    return (f'Cross-prototype plausible range with the target building '
            f'excluded: {round(lo, d)}\u2013{round(hi, d)}.')

def build_f1(building, climate):
    system = SYSTEM_PROMPT_BASE
    user = BASE_USER_PROMPT.format(building=building, location=CLIMATE_LABELS[climate])
    return system, user

def build_f2(building, climate):
    unit_lines = "\n".join(f'- {p["id"]}: {p["unit"]}' for p in PARAM_SCHEMA)
    system = SYSTEM_PROMPT_BASE + "\n\nUnits for each parameter:\n" + unit_lines
    user = BASE_USER_PROMPT.format(building=building, location=CLIMATE_LABELS[climate]) + UNITS_CLAUSE_TEXT
    return system, user

def build_f3(building, climate):
    lines = [f'- {p["id"]}: {_range_phrase(building, p)}' for p in PARAM_SCHEMA]
    system = SYSTEM_PROMPT_BASE + "\n\nCross-prototype plausible ranges (target building excluded):\n" + "\n".join(lines)
    user = BASE_USER_PROMPT.format(building=building, location=CLIMATE_LABELS[climate])
    return system, user

def build_f4(building, climate):
    lines = [f'- {p["id"]}: {p["unit"]} {_range_phrase(building, p)}' for p in PARAM_SCHEMA]
    system = SYSTEM_PROMPT_BASE + "\n\nUnits and cross-prototype plausible ranges (target building excluded):\n" + "\n".join(lines)
    user = BASE_USER_PROMPT.format(building=building, location=CLIMATE_LABELS[climate]) + UNITS_CLAUSE_TEXT
    return system, user

PROMPT_BUILDERS = {'F1': build_f1, 'F2': build_f2, 'F3': build_f3, 'F4': build_f4}

def prompt_hash(system, user):
    return hashlib.sha256((system + '\n' + user).encode()).hexdigest()[:16]

print(f'Part 1-A complete: {len(PARAM_SCHEMA)}-parameter schema and prompt builders defined.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
lobo_range_table.json SHA-256: 5229b2295c95c99b
LOBO range table integrity checks passed (bounds, SHGC, non-negativity, own-value exclusion, 9-param schema).
Part 1-A complete: 9-parameter schema and prompt builders defined.


In [7]:
# ============================================================
# PART 1-B — VALIDATE, SAVE, GATE: leakage check, format-rule
# check, cross-format consistency, then write prompt files.
# Stops (assert) if anything fails -- Part 2 will not run.
# ============================================================

UNIT_JARGON = ['W/m\u00b2K', 'W/m\u00b2', 'm\u00b2/person', 'm\u00b3/s', 'unitless',
               'degrees Celsius', 'fraction between 0 and 1', 'ratio (output/input)']
TRIVIAL_VALUES = {'0.0', '0', '1.0', '1', '-0.0'}

def validate_format_prompts(fmt, climates_to_run, ref_df_local):
    issues = []
    for building in TESTED_BUILDINGS:
        for climate in climates_to_run:
            system, user = PROMPT_BUILDERS[fmt](building, climate)
            schema_start = system.find('{"param_1"')
            schema_end = system.find('}', schema_start) + 1 if schema_start != -1 else -1
            text = (system[:schema_start] + system[schema_end:] + '\n' + user) if schema_start != -1 else system + '\n' + user

            raw_names = [p for p in PARAM_NAMES if p in text]
            if raw_names:
                issues.append(f'{fmt}/{building}/{climate}: raw parameter name(s) leaked: {raw_names}')

            control_leak = [p for p in CONTROL_PARAMS if p in text]
            if control_leak:
                issues.append(f'{fmt}/{building}/{climate}: control parameter name(s) leaked: {control_leak}')

            has_units = any(u in text for u in UNIT_JARGON)
            if fmt in ('F1', 'F3') and has_units:
                issues.append(f'{fmt}/{building}/{climate}: unit info present, should be absent')
            if fmt in ('F2', 'F4') and not has_units:
                issues.append(f'{fmt}/{building}/{climate}: unit info missing, should be present')

            has_ranges = bool(re.search(r'\d+\.?\d*\u2013\d+\.?\d*', text))
            if fmt in ('F1', 'F2') and has_ranges:
                issues.append(f'{fmt}/{building}/{climate}: numeric range present, should be absent')
            if fmt in ('F3', 'F4') and not has_ranges:
                issues.append(f'{fmt}/{building}/{climate}: numeric range missing, should be present')

            for col in PARAM_NAMES:
                val = ref_df_local.loc[building, col]
                try:
                    val_str = str(round(float(val), 6))
                except (TypeError, ValueError):
                    continue
                if val_str in TRIVIAL_VALUES:
                    continue
                if re.search(r'(?<![\d.])' + re.escape(val_str) + r'(?![\d])', text):
                    issues.append(f'{fmt}/{building}/{climate}: possible reference leak ({col}={val_str} appears verbatim)')
    return issues

def check_cross_format_consistency():
    issues = []
    for building in TESTED_BUILDINGS:
        _, u1 = build_f1(building, CLIMATES['main'])
        _, u2 = build_f2(building, CLIMATES['main'])
        _, u3 = build_f3(building, CLIMATES['main'])
        _, u4 = build_f4(building, CLIMATES['main'])
        if u1 != u3:
            issues.append(f'{building}: F1 vs F3 user_prompt differs beyond design factors')
        if u2.replace(UNITS_CLAUSE_TEXT, '') != u4.replace(UNITS_CLAUSE_TEXT, ''):
            issues.append(f'{building}: F2 vs F4 user_prompt differs beyond design factors')
        if u1 + UNITS_CLAUSE_TEXT != u2:
            issues.append(f'{building}: F1 vs F2 user_prompt differs beyond unit clause')
    return issues

ref_df = pd.read_csv(f'{SPRINT2_DIR}/reference_parameters.csv', index_col=0)
all_ok = True

for fmt in ['F1', 'F2', 'F3', 'F4']:
    climates_to_run = [CLIMATES['main']] + (CLIMATES['validation'] if fmt == 'F4' else [])
    records = []
    for building in TESTED_BUILDINGS:
        for climate in climates_to_run:
            system, user = PROMPT_BUILDERS[fmt](building, climate)
            records.append({'building': building, 'climate': climate,
                             'system_prompt': system, 'user_prompt': user})
    out_path = f'{PROMPTS_DIR}/{fmt.lower()}_prompts.json'
    with open(out_path, 'w') as f:
        json.dump(records, f, indent=2)

    issues = validate_format_prompts(fmt, climates_to_run, ref_df)
    if issues:
        all_ok = False
        print(f'[{fmt}] {len(records)} prompts saved to {out_path} -- {len(issues)} ISSUE(S):')
        for i in issues:
            print('   -', i)
    else:
        print(f'[{fmt}] {len(records)} prompts saved to {out_path} -- validation passed.')

cross_issues = check_cross_format_consistency()
if cross_issues:
    all_ok = False
    print(f'\nCross-format consistency FAILED -- {len(cross_issues)} issue(s):')
    for i in cross_issues:
        print('   -', i)
else:
    print('\nCross-format consistency passed: user_prompt identical across formats except the unit clause.')

assert all_ok, 'Sprint 3 prompt validation failed -- fix issues above before running Part 2.'
print('\nALL CHECKS PASSED — safe to proceed to Part 2.')

[F1] 5 prompts saved to /content/drive/MyDrive/BEM-LLM/analiz/sprint3_prompts/f1_prompts.json -- validation passed.
[F2] 5 prompts saved to /content/drive/MyDrive/BEM-LLM/analiz/sprint3_prompts/f2_prompts.json -- validation passed.
[F3] 5 prompts saved to /content/drive/MyDrive/BEM-LLM/analiz/sprint3_prompts/f3_prompts.json -- validation passed.
[F4] 15 prompts saved to /content/drive/MyDrive/BEM-LLM/analiz/sprint3_prompts/f4_prompts.json -- validation passed.

Cross-format consistency passed: user_prompt identical across formats except the unit clause.

ALL CHECKS PASSED — safe to proceed to Part 2.


## Part 2 — API Calls

Requires `GROQ_KEY_1` (and optionally `_2`, `_3`) in Colab Secrets.
Re-running is safe: each call is keyed on
`(building, climate, model, seed, prompt_hash)`, so completed
combinations are skipped and only missing or prompt-changed ones
are re-sent.

In [8]:
!pip install groq --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.1 MB/s eta 0:00:00


In [13]:
# ============================================================
# PART 2-A — API SETUP: keys, model resolution, seed
# repeatability, full call-settings log, environment record.
# (requires GROQ_KEY_1/2/3 in Colab Secrets)
# ============================================================

import time, sys, platform, math
from google.colab import userdata
from groq import Groq, RateLimitError, APIError

GROQ_KEY_ENV_NAMES = ['GROQ_KEY_1']
API_KEYS = [userdata.get(n) for n in GROQ_KEY_ENV_NAMES if userdata.get(n)]
assert API_KEYS, 'Add at least one GROQ_KEY_N in Colab Secrets.'
_clients = [Groq(api_key=k) for k in API_KEYS]
print(f'{len(API_KEYS)} API key(s) loaded.')

_any_calls_exist = any(os.path.exists(f'{PROMPTS_DIR}/{f}_calls.jsonl') for f in ['f1', 'f2', 'f3', 'f4'])

if 'models' in protocol and _any_calls_exist:
    MODELS = protocol['models']
    print(f'Models locked from a previous run: {MODELS}')
else:
    MODELS_PLANNED = ['llama-3.3-70b-versatile', 'openai/gpt-oss-120b']
    FALLBACK_MAP = {'llama-3.3-70b-versatile': 'llama-3.1-70b-versatile',
                     'openai/gpt-oss-120b': 'openai/gpt-oss-20b'}

    def _test_model(model_name):
        try:
            resp = _clients[0].chat.completions.create(
                model=model_name, messages=[{'role': 'user', 'content': 'Reply with the single word: OK'}],
                max_tokens=5)
            return True, resp.choices[0].message.content.strip()
        except Exception as e:
            return False, str(e)

    MODELS, resolution_log = [], []
    for planned in MODELS_PLANNED:
        ok, detail = _test_model(planned)
        if ok:
            MODELS.append(planned)
            resolution_log.append({'planned': planned, 'used': planned, 'status': 'ok'})
        else:
            fb = FALLBACK_MAP.get(planned)
            ok2, detail2 = (_test_model(fb) if fb else (False, 'no fallback defined'))
            if ok2:
                MODELS.append(fb)
                resolution_log.append({'planned': planned, 'used': fb, 'status': 'fallback', 'reason': detail})
            else:
                resolution_log.append({'planned': planned, 'used': None, 'status': 'unavailable', 'reason': detail})
        print(f'{planned}: {resolution_log[-1]["status"]}')
    assert MODELS, 'No model available; experiment cannot start.'
    protocol['models'] = MODELS
    protocol['model_resolution_log'] = resolution_log
    with open(f'{ANALIZ}/protocol.json', 'w') as f:
        json.dump(protocol, f, indent=2, default=str)
    print(f'Models locked: {MODELS}')

# Full call-settings log, including parameters left at default.
MODEL_SETTINGS = {}
for m in MODELS:
    if m.startswith('openai/gpt-oss'):
        MODEL_SETTINGS[m] = {
            'temperature': 0.7, 'max_tokens': 800, 'reasoning_effort': 'low', 'json_mode': True,
            'top_p': 1.0, 'frequency_penalty': 0.0, 'presence_penalty': 0.0,
            'response_format_type': 'json_object', 'timeout_s': 60,
        }
    else:
        MODEL_SETTINGS[m] = {
            'temperature': 0.7, 'max_tokens': 400, 'reasoning_effort': None, 'json_mode': True,
            'top_p': 1.0, 'frequency_penalty': 0.0, 'presence_penalty': 0.0,
            'response_format_type': 'json_object', 'timeout_s': 60,
        }
print('Model call settings (full):', json.dumps(MODEL_SETTINGS, indent=2))

protocol['retry_policy'] = {
    'max_attempts_per_call': len(_clients) * 2,
    'backoff_s': 3,
    'consecutive_failure_stop_threshold': 5,
}
with open(f'{ANALIZ}/protocol.json', 'w') as f:
    json.dump(protocol, f, indent=2, default=str)

pkg_versions = {pkg: __import__(pkg).__version__ for pkg in ['numpy', 'pandas', 'groq']}
with open(f'{DATA}/environment_sprint3.json', 'w') as f:
    json.dump({'sprint': 'Sprint 3', 'python_version': sys.version, 'platform': platform.platform(),
               'package_versions': pkg_versions, 'master_seed': MASTER_SEED,
               'n_api_keys': len(API_KEYS), 'models': MODELS,
               'model_settings': MODEL_SETTINGS}, f, indent=2)
print(f'Saved: {DATA}/environment_sprint3.json')

# --- Rotating client: returns the actual retry/attempt count ---
_key_idx = 0
def call_with_rotation(model, system, user, seed):
    global _key_idx
    settings = MODEL_SETTINGS[model]
    kwargs = dict(model=model, messages=[{'role': 'system', 'content': system}, {'role': 'user', 'content': user}],
                  max_tokens=settings['max_tokens'], temperature=settings['temperature'], seed=seed,
                  top_p=settings['top_p'], frequency_penalty=settings['frequency_penalty'],
                  presence_penalty=settings['presence_penalty'], timeout=settings['timeout_s'])
    if settings.get('json_mode'):
        kwargs['response_format'] = {'type': settings['response_format_type']}
    if settings.get('reasoning_effort'):
        kwargs['reasoning_effort'] = settings['reasoning_effort']
    attempts, max_attempts = 0, len(_clients) * 2
    while attempts < max_attempts:
        client = _clients[_key_idx % len(_clients)]
        try:
            resp = client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content.strip(), _key_idx % len(_clients), settings, attempts
        except RateLimitError:
            print(f'    key {_key_idx % len(_clients)} rate limited, rotating')
            _key_idx += 1; attempts += 1
        except APIError as e:
            print(f'    API error on key {_key_idx % len(_clients)}: {e}, rotating')
            _key_idx += 1; attempts += 1
            time.sleep(3)
    raise RuntimeError('All API keys exhausted or repeatedly failing.')

# --- JSON parser: reports whether a duplicate key was present ---
def parse_json(raw):
    raw = re.sub(r'```[a-zA-Z]*\n?', '', raw).strip().replace('```', '')
    s, e = raw.find('{'), raw.rfind('}')
    if s == -1 or e == -1:
        return None, False
    dup_found = [False]
    def _hook(pairs):
        seen = set()
        for k, _ in pairs:
            if k in seen:
                dup_found[0] = True
            seen.add(k)
        return dict(pairs)
    try:
        return json.loads(raw[s:e + 1], object_pairs_hook=_hook), dup_found[0]
    except Exception:
        return None, False

# --- Strict schema check: exact key set, no duplicates, numeric, finite ---
def schema_check(parsed, has_duplicates):
    if parsed is None:
        return False, 'parse_failed'
    if has_duplicates:
        return False, 'duplicate_key'
    keys = set(parsed.keys())
    required = set(JSON_KEY_ORDER)
    if keys != required:
        extra = keys - required
        missing = required - keys
        return False, f'key_mismatch(extra={list(extra)}, missing={list(missing)})'
    for k, v in parsed.items():
        if not isinstance(v, (int, float)) or isinstance(v, bool):
            return False, f'{k}_not_numeric'
        if not math.isfinite(v):
            return False, f'{k}_not_finite'
    return True, 'ok'

PHYSICAL_BOUNDS = dict(protocol['physical_bounds'])

def check_bounds(parsed):
    violations = []
    if not parsed:
        return violations
    for pid, val in parsed.items():
        name = ID_TO_NAME.get(pid)
        bounds = PHYSICAL_BOUNDS.get(name)
        if bounds and isinstance(val, (int, float)):
            lo, hi = bounds
            if (lo is not None and val < lo) or (hi is not None and val > hi):
                violations.append({'parameter': name, 'value': val, 'bounds': bounds})
    return violations

for i, seed in enumerate(SEEDS):
    sent_seeds = {m: seed for m in MODELS}
    assert len(set(sent_seeds.values())) == 1, \
        f'Seed index {i}: models were not sent an identical seed value'
print(f'Common-seed check passed: all {len(MODELS)} models receive identical seed at each of {len(SEEDS)} seed indices.')

# --- Seed repeatability smoke test using real experimental prompts, not a
# trivial fixed-answer probe. Cached to seed_repeatability.json so re-running
# this cell does NOT repeat the 42 API calls if already completed. ---
SEED_TEST_PATH = f'{PROMPTS_DIR}/seed_repeatability.json'

def _seed_repeatability_test(model, n_prompts=3, same_seed_repeats=3):
    sample_buildings = TESTED_BUILDINGS[:n_prompts]
    results = []
    for building in sample_buildings:
        system, user = PROMPT_BUILDERS['F4'](building, CLIMATES['main'])
        print(f'    [{model}] {building}: same-seed x{same_seed_repeats}...', end=' ', flush=True)

        same_seed_outputs = []
        for i in range(same_seed_repeats):
            same_seed_outputs.append(call_with_rotation(model, system, user, SEEDS[0])[0])
            print(i + 1, end=' ', flush=True)
        repeatable = len(set(same_seed_outputs)) == 1
        print(f'-> repeatable={repeatable}')

        print(f'    [{model}] {building}: cross-seed x{len(SEEDS) - 1}...', end=' ', flush=True)
        diff_seed_outputs = {}
        for s in SEEDS[1:]:
            diff_seed_outputs[s] = call_with_rotation(model, system, user, s)[0]
            print(s, end=' ', flush=True)
        distinct_across_seeds = len(set(diff_seed_outputs.values())) > 1
        print(f'-> distinct={distinct_across_seeds}')

        results.append({'building': building, 'same_seed_repeatable': repeatable,
                         'distinct_across_seeds': distinct_across_seeds})
    return results

if os.path.exists(SEED_TEST_PATH):
    seed_repeatability = json.load(open(SEED_TEST_PATH))
    print(f'Seed repeatability already cached in {SEED_TEST_PATH}, skipping API calls.')
else:
    seed_repeatability = {}
    for m in MODELS:
        print(f'Running seed repeatability test for {m} (21 calls: 3 buildings x (3 same-seed + 4 cross-seed))...')
        seed_repeatability[m] = _seed_repeatability_test(m)
    with open(SEED_TEST_PATH, 'w') as f:
        json.dump(seed_repeatability, f, indent=2)

for m, res in seed_repeatability.items():
    n_repeat = sum(r['same_seed_repeatable'] for r in res)
    n_distinct = sum(r['distinct_across_seeds'] for r in res)
    print(f'{m}: same-seed repeatability {n_repeat}/{len(res)}; distinct outputs across seeds {n_distinct}/{len(res)}')

protocol['seed_control'] = {
    'method': 'same-seed repeatability smoke test / provider-supported best-effort seed control',
    'results_by_model': seed_repeatability,
}
with open(f'{ANALIZ}/protocol.json', 'w') as f:
    json.dump(protocol, f, indent=2, default=str)

print('\nPart 2-A complete: models, settings, client, and validation helpers ready.')

1 API key(s) loaded.
Models locked from a previous run: ['llama-3.3-70b-versatile', 'openai/gpt-oss-120b']
Model call settings (full): {
  "llama-3.3-70b-versatile": {
    "temperature": 0.7,
    "max_tokens": 400,
    "reasoning_effort": null,
    "json_mode": true,
    "top_p": 1.0,
    "frequency_penalty": 0.0,
    "presence_penalty": 0.0,
    "response_format_type": "json_object",
    "timeout_s": 60
  },
  "openai/gpt-oss-120b": {
    "temperature": 0.7,
    "max_tokens": 800,
    "reasoning_effort": "low",
    "json_mode": true,
    "top_p": 1.0,
    "frequency_penalty": 0.0,
    "presence_penalty": 0.0,
    "response_format_type": "json_object",
    "timeout_s": 60
  }
}
Saved: /content/drive/MyDrive/BEM-LLM/data/environment_sprint3.json
Common-seed check passed: all 2 models receive identical seed at each of 5 seed indices.
Seed repeatability already cached in /content/drive/MyDrive/BEM-LLM/analiz/sprint3_prompts/seed_repeatability.json, skipping API calls.
llama-3.3-70b-versat

In [14]:
# ============================================================
# PART 2-B — RUN ALL FORMATS, AGGREGATE, REPORT
# Resumability keys on (building, climate, model, seed,
# prompt_hash). Stops after 5 consecutive failures. Logs the
# real retry_count, strict schema check (incl. duplicate keys).
# ============================================================

def run_format(fmt, extra_climates=None):
    climates_to_run = [CLIMATES['main']] + (extra_climates or [])
    out_path = f'{PROMPTS_DIR}/{fmt.lower()}_calls.jsonl'
    fmt_start = time.perf_counter()

    done_keys = set()
    if os.path.exists(out_path):
        with open(out_path) as f:
            for line in f:
                r = json.loads(line)
                if r.get('parse_ok') and r.get('complete_ok'):
                    done_keys.add((r['building'], r['climate'], r['model'], r['seed'], r.get('prompt_hash')))
    print(f'[{fmt}] {len(done_keys)} combinations already completed with current prompts, skipping.')

    combos = [(b, c, m, s) for b in TESTED_BUILDINGS for c in climates_to_run for m in MODELS for s in SEEDS]
    consecutive_failures = 0

    with open(out_path, 'a') as fout:
        for building, climate, model, seed in combos:
            system, user = PROMPT_BUILDERS[fmt](building, climate)
            phash = prompt_hash(system, user)
            key = (building, climate, model, seed, phash)
            if key in done_keys:
                continue
            t0 = time.perf_counter()
            try:
                raw, key_used, settings, retry_count = call_with_rotation(model, system, user, seed)
            except Exception as e:
                consecutive_failures += 1
                error_record = {'building': building, 'climate': climate, 'format': fmt, 'model': model,
                                 'seed': seed, 'status': 'api_error', 'error_reason': str(e),
                                 'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S')}
                with open(f'{PROMPTS_DIR}/error_log.jsonl', 'a') as ferr:
                    ferr.write(json.dumps(error_record) + '\n')
                print(f'  FAILED {key}: {e}')
                if consecutive_failures >= 5:
                    raise RuntimeError('5 consecutive failures -- stopping. Check API keys/settings.')
                continue
            consecutive_failures = 0
            latency = round(time.perf_counter() - t0, 2)
            parsed, has_duplicates = parse_json(raw)
            parse_ok = parsed is not None
            complete_ok, schema_reason = schema_check(parsed, has_duplicates)
            violations = check_bounds(parsed)
            record = {'building': building, 'climate': climate, 'format': fmt, 'model': model, 'seed': seed,
                      'api_key_index': key_used, 'prompt_hash': phash, 'retry_count': retry_count,
                      'temperature': settings['temperature'], 'max_tokens': settings['max_tokens'],
                      'top_p': settings['top_p'], 'frequency_penalty': settings['frequency_penalty'],
                      'presence_penalty': settings['presence_penalty'], 'timeout_s': settings['timeout_s'],
                      'reasoning_effort': settings.get('reasoning_effort'),
                      'json_mode': settings.get('json_mode', False),
                      'response_format_type': settings.get('response_format_type'),
                      'system_prompt': system, 'user_prompt': user,
                      'raw_response': raw, 'parsed': parsed, 'parse_ok': parse_ok,
                      'complete_ok': complete_ok, 'schema_reason': schema_reason,
                      'bound_violations': violations,
                      'latency_s': latency, 'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S')}
            fout.write(json.dumps(record) + '\n')
            fout.flush()
            status = 'OK' if complete_ok else schema_reason
            viol_note = f' bounds_violated={len(violations)}' if violations else ''
            print(f'  {building:<20} {climate} {model:<28} seed={seed}  {status}  retries={retry_count}{viol_note}  ({latency:.2f}s)')
            time.sleep(2)

    fmt_duration = round(time.perf_counter() - fmt_start, 1)
    print(f'[{fmt}] Saved: {out_path}  (format duration: {fmt_duration}s = {fmt_duration/60:.1f} min)')
    return fmt_duration

run_start = time.perf_counter()
run_timestamp_start = time.strftime('%Y-%m-%dT%H:%M:%S')

duration_f1 = run_format('F1')
duration_f2 = run_format('F2')
duration_f3 = run_format('F3')
duration_f4 = run_format('F4', extra_climates=CLIMATES['validation'])

total_duration = round(time.perf_counter() - run_start, 1)
run_timestamp_end = time.strftime('%Y-%m-%dT%H:%M:%S')
print(f'\nTotal API run duration: {total_duration:.1f}s = {total_duration/60:.1f} min = {total_duration/3600:.2f} hr')

run_duration_record = {
    'run_started': run_timestamp_start,
    'run_ended': run_timestamp_end,
    'total_duration_s': total_duration,
    'per_format_duration_s': {'F1': duration_f1, 'F2': duration_f2, 'F3': duration_f3, 'F4': duration_f4},
}
with open(f'{PROMPTS_DIR}/run_duration.json', 'w') as f:
    json.dump(run_duration_record, f, indent=2)
print(f'Saved: {PROMPTS_DIR}/run_duration.json')

# --- Aggregate all calls, report completeness, save missing combinations ---
all_records = []
for fmt in ['F1', 'F2', 'F3', 'F4']:
    path = f'{PROMPTS_DIR}/{fmt.lower()}_calls.jsonl'
    if os.path.exists(path):
        with open(path) as f:
            all_records += [json.loads(line) for line in f]

calls_df = pd.DataFrame(all_records)
print(f'\nTotal calls: {len(calls_df)}')
if not calls_df.empty:
    print(f'Parse success rate: {calls_df.parse_ok.mean()*100:.1f}%')
    print(f'Schema-complete (exact key set, no duplicates, numeric, finite) rate: {calls_df.complete_ok.mean()*100:.1f}%')
    print(f'Mean retries per call: {calls_df.retry_count.mean():.2f}')
    print(calls_df.groupby(['format', 'climate']).size().to_string())
    if 'schema_reason' in calls_df.columns:
        print('\nSchema failure reasons (excluding "ok"):')
        print(calls_df[calls_df.schema_reason != 'ok'].schema_reason.value_counts().to_string())

bound_rows = []
for r in all_records:
    for v in r.get('bound_violations', []):
        bound_rows.append({'parameter': v['parameter'], 'model': r['model'], 'format': r['format'],
                            'value': v['value'], 'bounds': v['bounds']})
if bound_rows:
    bound_df = pd.DataFrame(bound_rows)
    summary = bound_df.groupby(['parameter', 'model']).size().reset_index(name='count')
    summary.to_csv(f'{PROMPTS_DIR}/bound_violation_log.csv', index=False)
    print(f'\n{len(bound_rows)} physical bound violation(s) found. Saved: {PROMPTS_DIR}/bound_violation_log.csv')
else:
    pd.DataFrame(columns=['parameter', 'model', 'count']).to_csv(f'{PROMPTS_DIR}/bound_violation_log.csv', index=False)
    print('\nNo physical bound violations found.')

missing = []
for fmt in ['F1', 'F2', 'F3']:
    for b in TESTED_BUILDINGS:
        for m in MODELS:
            for s in SEEDS:
                match = calls_df[(calls_df.format == fmt) & (calls_df.building == b) &
                                  (calls_df.climate == CLIMATES['main']) & (calls_df.model == m) &
                                  (calls_df.seed == s) & calls_df.complete_ok] if not calls_df.empty else pd.DataFrame()
                if match.empty:
                    missing.append({'format': fmt, 'building': b, 'climate': CLIMATES['main'], 'model': m, 'seed': s})
for c in [CLIMATES['main']] + CLIMATES['validation']:
    for b in TESTED_BUILDINGS:
        for m in MODELS:
            for s in SEEDS:
                match = calls_df[(calls_df.format == 'F4') & (calls_df.building == b) & (calls_df.climate == c) &
                                  (calls_df.model == m) & (calls_df.seed == s) & calls_df.complete_ok] if not calls_df.empty else pd.DataFrame()
                if match.empty:
                    missing.append({'format': 'F4', 'building': b, 'climate': c, 'model': m, 'seed': s})

pd.DataFrame(missing).to_csv(f'{PROMPTS_DIR}/missing_combinations.csv', index=False)
print(f'{len(missing)} missing combination(s). Saved: {PROMPTS_DIR}/missing_combinations.csv')

debug_rows = [{'building': r['building'], 'format': r['format'], 'model': r['model'], 'seed': r['seed'],
               'climate': r['climate'], 'param_id': pid, 'true_name': ID_TO_NAME.get(pid, 'UNKNOWN'), 'value': val}
              for r in all_records if r['parsed'] for pid, val in r['parsed'].items()]
pd.DataFrame(debug_rows).to_csv(f'{PROMPTS_DIR}/param_debug_mapping.csv', index=False)
print(f'Saved: {PROMPTS_DIR}/param_debug_mapping.csv')

[F1] 50 combinations already completed with current prompts, skipping.
[F1] Saved: /content/drive/MyDrive/BEM-LLM/analiz/sprint3_prompts/f1_calls.jsonl  (format duration: 0.0s = 0.0 min)
[F2] 50 combinations already completed with current prompts, skipping.
[F2] Saved: /content/drive/MyDrive/BEM-LLM/analiz/sprint3_prompts/f2_calls.jsonl  (format duration: 0.0s = 0.0 min)
[F3] 100 combinations already completed with current prompts, skipping.
[F3] Saved: /content/drive/MyDrive/BEM-LLM/analiz/sprint3_prompts/f3_calls.jsonl  (format duration: 0.0s = 0.0 min)
[F4] 236 combinations already completed with current prompts, skipping.
  RetailStripmall      7 llama-3.3-70b-versatile      seed=1542799867  OK  retries=0  (0.42s)
  RetailStripmall      7 llama-3.3-70b-versatile      seed=1542799868  OK  retries=0  (0.32s)
  RetailStripmall      7 llama-3.3-70b-versatile      seed=1542799869  OK  retries=0  (0.69s)
  RetailStripmall      7 llama-3.3-70b-versatile      seed=1542799870  OK  retries=0

In [16]:
# Part 2-B-CLEANUP  Prunes stale rows left behind by a previous prompt
# version (e.g. the LOBO-range fix), then regenerates every downstream file
# from the cleaned calls so no analysis output mixes old and new prompts.
# No API calls -- reads existing *_calls.jsonl and uses the in-memory
# PROMPT_BUILDERS to recompute each row's *current* prompt_hash.

# --- Step 1: prune stale (superseded-prompt) rows from each calls.jsonl ---
cleaned_all_records = []
for fmt in ['F1', 'F2', 'F3', 'F4']:
    path = f'{PROMPTS_DIR}/{fmt.lower()}_calls.jsonl'
    if not os.path.exists(path):
        continue

    current_hash = {}
    for building in TESTED_BUILDINGS:
        for climate in [CLIMATES['main']] + CLIMATES['validation']:
            system, user = PROMPT_BUILDERS[fmt](building, climate)
            current_hash[(building, climate)] = prompt_hash(system, user)

    with open(path) as f:
        records = [json.loads(line) for line in f]

    kept = [r for r in records if r.get('prompt_hash') == current_hash.get((r['building'], r['climate']))]
    n_dropped = len(records) - len(kept)
    print(f'[{fmt}] {len(records)} -> {len(kept)} (pruned {n_dropped} stale row(s) from a superseded prompt version)')

    with open(path, 'w') as f:
        for r in kept:
            f.write(json.dumps(r) + '\n')

    cleaned_all_records += kept

print(f'\nTotal clean records across all formats: {len(cleaned_all_records)}')

# --- Step 2: regenerate param_debug_mapping.csv ---
debug_rows = [{'building': r['building'], 'format': r['format'], 'model': r['model'], 'seed': r['seed'],
               'climate': r['climate'], 'param_id': pid, 'true_name': ID_TO_NAME.get(pid, 'UNKNOWN'), 'value': val}
              for r in cleaned_all_records if r.get('parsed') for pid, val in r['parsed'].items()]
pd.DataFrame(debug_rows).to_csv(f'{PROMPTS_DIR}/param_debug_mapping.csv', index=False)
print(f'Saved: param_debug_mapping.csv ({len(debug_rows)} rows)')

# --- Step 3: regenerate bound_violation_log.csv ---
bound_rows = []
for r in cleaned_all_records:
    for v in r.get('bound_violations', []):
        bound_rows.append({'parameter': v['parameter'], 'model': r['model'], 'format': r['format'],
                            'value': v['value'], 'bounds': v['bounds']})
if bound_rows:
    bound_df = pd.DataFrame(bound_rows)
    summary = bound_df.groupby(['parameter', 'model']).size().reset_index(name='count')
    summary.to_csv(f'{PROMPTS_DIR}/bound_violation_log.csv', index=False)
else:
    pd.DataFrame(columns=['parameter', 'model', 'count']).to_csv(f'{PROMPTS_DIR}/bound_violation_log.csv', index=False)
print(f'Saved: bound_violation_log.csv ({len(bound_rows)} raw violations)')

# --- Step 4: regenerate missing_combinations.csv ---
calls_df = pd.DataFrame(cleaned_all_records)
missing = []
for fmt in ['F1', 'F2', 'F3']:
    for b in TESTED_BUILDINGS:
        for m in MODELS:
            for s in SEEDS:
                match = calls_df[(calls_df.format == fmt) & (calls_df.building == b) &
                                  (calls_df.climate == CLIMATES['main']) & (calls_df.model == m) &
                                  (calls_df.seed == s) & calls_df.complete_ok]
                if match.empty:
                    missing.append({'format': fmt, 'building': b, 'climate': CLIMATES['main'], 'model': m, 'seed': s})
for c in [CLIMATES['main']] + CLIMATES['validation']:
    for b in TESTED_BUILDINGS:
        for m in MODELS:
            for s in SEEDS:
                match = calls_df[(calls_df.format == 'F4') & (calls_df.building == b) & (calls_df.climate == c) &
                                  (calls_df.model == m) & (calls_df.seed == s) & calls_df.complete_ok]
                if match.empty:
                    missing.append({'format': 'F4', 'building': b, 'climate': c, 'model': m, 'seed': s})
pd.DataFrame(missing).to_csv(f'{PROMPTS_DIR}/missing_combinations.csv', index=False)
print(f'Saved: missing_combinations.csv ({len(missing)} missing)')

# --- Step 5: regenerate range_adherence_log.csv (Part 2-C, madde 3) ---
adherence_rows = []
for r in cleaned_all_records:
    if not r.get('parsed'):
        continue
    for pid, val in r['parsed'].items():
        name = ID_TO_NAME.get(pid)
        row = {'building': r['building'], 'format': r['format'], 'model': r['model'],
               'seed': r['seed'], 'climate': r['climate'], 'climate_role': CLIMATE_ROLE.get(r['climate']),
               'parameter': name, 'value': val}
        bounds = PHYSICAL_BOUNDS.get(name)
        if bounds:
            lo, hi = bounds
            row['physical_bound_ok'] = not ((lo is not None and val < lo) or (hi is not None and val > hi))
        else:
            row['physical_bound_ok'] = None
        if r['format'] in ('F3', 'F4') and name in RANGE_TABLE.get(r['building'], {}):
            lo_r, hi_r = RANGE_TABLE[r['building']][name]
            row['lobo_range_ok'] = lo_r <= val <= hi_r
        else:
            row['lobo_range_ok'] = None
        adherence_rows.append(row)

adherence_df = pd.DataFrame(adherence_rows)
adherence_df.to_csv(f'{PROMPTS_DIR}/range_adherence_log.csv', index=False)
n_range_violations = (adherence_df['lobo_range_ok'] == False).sum()
print(f'Saved: range_adherence_log.csv ({len(adherence_df)} rows, {n_range_violations} LOBO-range violations)')
if n_range_violations:
    print(adherence_df[adherence_df['lobo_range_ok'] == False].groupby(['format', 'model', 'climate']).size().to_string())

# --- Step 6: regenerate fahrenheit_diagnostic_table.csv (Part 2-C) ---
def f_to_c(f):
    return (f - 32) * 5 / 9

setpoints = adherence_df[adherence_df['parameter'].isin(['heating_setpoint_C', 'cooling_setpoint_C'])].copy()
setpoints['looks_fahrenheit'] = setpoints['value'].between(60, 85)
setpoints['converted_C'] = setpoints['value'].apply(f_to_c)
setpoints['converted_in_bounds'] = setpoints['converted_C'].between(10, 30)

fahrenheit_diag = (setpoints.groupby(['model', 'format', 'parameter'])
                   .agg(n_calls=('value', 'size'), n_fahrenheit_like=('looks_fahrenheit', 'sum'),
                        n_would_be_valid_after_conversion=('converted_in_bounds', 'sum'))
                   .reset_index())
fahrenheit_diag.to_csv(f'{PROMPTS_DIR}/fahrenheit_diagnostic_table.csv', index=False)
print(f'\nSaved: fahrenheit_diagnostic_table.csv')
print(fahrenheit_diag.to_string(index=False))

[F1] 50 -> 50 (pruned 0 stale row(s) from a superseded prompt version)
[F2] 50 -> 50 (pruned 0 stale row(s) from a superseded prompt version)
[F3] 100 -> 50 (pruned 50 stale row(s) from a superseded prompt version)
[F4] 300 -> 150 (pruned 150 stale row(s) from a superseded prompt version)

Total clean records across all formats: 300
Saved: param_debug_mapping.csv (2700 rows)
Saved: bound_violation_log.csv (48 raw violations)
Saved: missing_combinations.csv (0 missing)
Saved: range_adherence_log.csv (2700 rows, 5 LOBO-range violations)
format  model                    climate
F3      llama-3.3-70b-versatile  5A         4
F4      llama-3.3-70b-versatile  1A         1

Saved: fahrenheit_diagnostic_table.csv
                  model format          parameter  n_calls  n_fahrenheit_like  n_would_be_valid_after_conversion
llama-3.3-70b-versatile     F1 cooling_setpoint_C       25                 24                                 24
llama-3.3-70b-versatile     F1 heating_setpoint_C       25  

## F3/F4 Data Provenance Cleanup

After a fenestration-area indexing bug was found and corrected during Sprint 2's manual verification, the LOBO range for two parameters (`window_u_value`, `window_shgc`) changed. Since F3/F4 prompts embed this range as text, their prompt content changed too -- but `run_format`'s append-only write meant `f3_calls.jsonl`/`f4_calls.jsonl` temporarily held **both the pre-fix and post-fix generations** under the same `(building, climate, model, seed)` label.

**Cleanup criterion (mechanical, not outcome-based):** each row's `prompt_hash` was compared against the hash `PROMPT_BUILDERS` currently produces for that `(building, climate)`; rows not matching the current protocol were dropped. No response was excluded based on its value -- only rows belonging to a superseded prompt version were removed. F1/F2 prompts carry no range text and were unaffected (0 rows pruned in both).

**Result:** F3 100→**50** rows, F4 300→**150** rows; all 300 combinations remain complete (`missing_combinations.csv` empty). Recomputed on the clean set: `bound_violation_log.csv` unchanged (48 violations, all F1/Llama); `range_adherence_log.csv` now shows **5** LOBO-range violations -- F3/Llama/5A: 4 (new finding: the narrower corrected `window_shgc` range now excludes some previously-compliant Llama outputs), F4/Llama/1A: 1 (known weather-file stress-test violation). Violations are kept raw, not corrected or clipped. The cleanup step requires no API calls and is fully reproducible from the hash comparison alone.

**Manuscript sentence:** *"During manual verification, an indexing error in the window-area calculation was identified and corrected, altering the LOBO range for two parameters (window U-value, window SHGC). F3/F4 prompts embedding these ranges were regenerated accordingly; calls generated under the superseded range were excluded from analysis via prompt-hash matching. F1/F2 prompts do not embed range information and were unaffected."*

**(TR)**

## F3/F4 Veri Kökeni Temizliği

Sprint 2'nin manuel doğrulaması sırasında bulunan bir fenestrasyon-alanı indeks hatası düzeltilince, iki parametrenin (`window_u_value`, `window_shgc`) LOBO aralığı değişti. F3/F4 promptları bu aralığı metne gömdüğü için prompt içeriği de değişti -- ama `run_format`'ın ekleme-modlu (append-only) yazması nedeniyle `f3_calls.jsonl`/`f4_calls.jsonl` bir süre **hem düzeltme öncesi hem sonrası üretimi** aynı `(bina, iklim, model, tohum)` etiketi altında karışık tutuyordu.

**Temizlik kriteri (mekanik, sonuca bakılmadan):** her satırın `prompt_hash`'i, `PROMPT_BUILDERS`'ın o `(bina, iklim)` için **şu an** ürettiği hash ile karşılaştırıldı; güncel protokole ait olmayan satırlar atıldı. Hiçbir cevap, değerine bakılarak elenmedi -- yalnızca eski bir prompt sürümüne ait satırlar çıkarıldı. F1/F2 hiç aralık metni içermediği için etkilenmedi (ikisinde de 0 satır atıldı).

**Sonuç:** F3 100→**50** satır, F4 300→**150** satır; 300 kombinasyonun hepsi tam kaldı (`missing_combinations.csv` boş). Temiz veriyle yeniden hesaplandığında: `bound_violation_log.csv` değişmedi (48 ihlal, hepsi F1/Llama); `range_adherence_log.csv` artık **5** LOBO-aralık ihlali gösteriyor -- F3/Llama/5A: 4 (yeni bulgu: daraltılmış doğru `window_shgc` aralığı, önceden uyumlu görünen bazı Llama çıktılarını artık dışarıda bırakıyor), F4/Llama/1A: 1 (bilinen weather-file stress-test ihlali). İhlaller düzeltilmeden/kırpılmadan ham tutuluyor. Bu temizlik adımı hiçbir API çağrısı gerektirmiyor ve yalnızca hash karşılaştırmasından tamamen yeniden üretilebilir.

**Makale cümlesi:** *"Manuel doğrulama sırasında pencere-alanı hesabında bir indeks hatası bulunup düzeltildi, bu da iki parametrenin (pencere U-değeri, SHGC) LOBO aralığını değiştirdi. Bu aralıkları içeren F3/F4 promptları buna göre yeniden üretildi; eski aralığa ait çağrılar prompt-hash eşleşmesiyle analizden çıkarıldı. F1/F2 promptları aralık bilgisi içermediği için etkilenmedi."*

In [17]:
# Sprint 3 — Final Audit: expected vs actual prompt/response counts (no API calls)
import json

EXPECTED = {
    'F1': len(TESTED_BUILDINGS) * len(MODELS) * len(SEEDS),                                   # 5x2x5 = 50
    'F2': len(TESTED_BUILDINGS) * len(MODELS) * len(SEEDS),                                   # 50
    'F3': len(TESTED_BUILDINGS) * len(MODELS) * len(SEEDS),                                   # 50
    'F4': len(TESTED_BUILDINGS) * len(MODELS) * len(SEEDS) * (1 + len(CLIMATES['validation'])),  # 5x2x5x3 = 150
}

print(f'{"Format":<6} {"Prompts":>8} {"Responses":>10} {"parse_ok":>9} {"complete_ok":>12} {"Expected":>9}  Status')
print('-' * 65)
for fmt in ['F1', 'F2', 'F3', 'F4']:
    with open(f'{PROMPTS_DIR}/{fmt.lower()}_prompts.json') as f:
        n_prompts = len(json.load(f))
    with open(f'{PROMPTS_DIR}/{fmt.lower()}_calls.jsonl') as f:
        recs = [json.loads(l) for l in f]
    n_resp = len(recs)
    n_parse = sum(r.get('parse_ok', False) for r in recs)
    n_complete = sum(r.get('complete_ok', False) for r in recs)
    exp = EXPECTED[fmt]
    ok = (n_resp == exp == n_parse == n_complete)
    print(f'{fmt:<6} {n_prompts:>8} {n_resp:>10} {n_parse:>9} {n_complete:>12} {exp:>9}  {"OK" if ok else "CHECK"}')

# F4 breakdown by climate -- confirms the 5A/1A/7 split is exactly 50/50/50
print('\nF4 breakdown by climate:')
with open(f'{PROMPTS_DIR}/f4_calls.jsonl') as f:
    f4_recs = [json.loads(l) for l in f]
for c in [CLIMATES['main']] + CLIMATES['validation']:
    sub = [r for r in f4_recs if r['climate'] == c]
    exp_c = len(TESTED_BUILDINGS) * len(MODELS) * len(SEEDS)
    print(f'  {c:<4} {len(sub)}/{exp_c}  parse_ok={sum(r["parse_ok"] for r in sub)}  complete_ok={sum(r["complete_ok"] for r in sub)}')

print(f'\nTotal calls across F1-F4: {sum(EXPECTED.values())} expected, '
      f'{sum(len([json.loads(l) for l in open(f"{PROMPTS_DIR}/{f.lower()}_calls.jsonl")]) for f in ["F1","F2","F3","F4"])} actual.')

Format  Prompts  Responses  parse_ok  complete_ok  Expected  Status
-----------------------------------------------------------------
F1            5         50        50           50        50  OK
F2            5         50        50           50        50  OK
F3            5         50        50           50        50  OK
F4           15        150       150          150       150  OK

F4 breakdown by climate:
  5A   50/50  parse_ok=50  complete_ok=50
  1A   50/50  parse_ok=50  complete_ok=50
  7    50/50  parse_ok=50  complete_ok=50

Total calls across F1-F4: 300 expected, 300 actual.
